In [0]:
--------------------------------------------------------------------------------------------------------------
-- © Tomasz Falkowski 2026
--------------------------------------------------------------------------------------------------------------
-- INSTRUCTIONS
-- Each section (1.1, 1.2. etc.) can be run independently based on provided tables: 'transactions' and 'contact_history', however it may require the temporary view 'promo'. Please run the code in section 0. before proceeding with queries in other sections.
--------------------------------------------------------------------------------------------------------------
-- 0. Create temporary view 'promo':
    -- - Table contact_history,
    -- - Left joined with transactions table on customer id,
    -- - Filtered for the promotional period date range,
    -- - Grouped up customers by id in order to manage multiple purchases made by the same customer during the promotional period,
    -- - Dates were grouped up by picking the most recent purchase for given customer,
    -- - Summing up all amount purchases made by the same customer.
    CREATE OR REPLACE VIEW promo AS (
        -- 0.1. CTE 'ch_t_joined_and_date_filtered' with both tables joined by customer id and filtered for the promotional period date range
        WITH ch_t_joined_and_date_filtered AS (  
            SELECT 
                ch.customer_id,
                ch.group,
                t.purchase_date,
                t.purchase_amount
            FROM contact_history as ch
            -- Left join in order to receive null values for customers who did not make any purchases at all
            LEFT JOIN transactions as t
                ON ch.customer_id = t.customer_id
            -- Applying promotional period date range
            WHERE t.purchase_date BETWEEN '2024-02-01' AND '2024-02-14'
        )
        -- 0.2. Grouped up customers by id in order to manage multiple purchases made by the same customer during the promotional period
        SELECT 
            customer_id,
            group,
            -- Dates were grouped up by picking the most recent purchase for the same customer
            MAX(purchase_date) as max_purchase_date,
            -- Summing up all amount purchases made by the same customer
            SUM(purchase_amount) as sum_purchase_amount
        FROM ch_t_joined_and_date_filtered
        GROUP BY customer_id, group
        ORDER BY max_purchase_date, customer_id
    );

--------------------------------------------------------------------------------------------------------------
-- 1. Overview
-- 1.1 Participants of Promotional Campaign by Group
    SELECT group, COUNT(DISTINCT customer_id) as participants
    FROM contact_history
    GROUP BY group
    ORDER BY participants DESC;

-- 1.2 All Distinct Customers during Promotional Period
    SELECT COUNT(DISTINCT customer_id) as total_distinct_customers
    FROM transactions
    WHERE purchase_date BETWEEN '2024-02-01' AND '2024-02-14';

-- 1.3 Purchase Frequency during Promotional Period
    SELECT COUNT(DISTINCT customer_id) as total_distinct_customers,
        purchase_date
    FROM transactions
    WHERE purchase_date BETWEEN '2024-02-01' AND '2024-02-14'
    GROUP BY purchase_date
    ORDER BY purchase_date;

-- 1.4 Online Purchase by All Customers
    SELECT t.purchase_date,
        dayname(t.purchase_date) AS day_of_week,
        SUM(CASE WHEN p.group = 'Sent' THEN t.purchase_amount 
                ELSE 0 END) AS sent_amount,
        SUM(CASE WHEN p.group = 'Control' THEN t.purchase_amount 
                ELSE 0 END) AS control_amount,
        SUM(CASE WHEN p.group IS NULL THEN t.purchase_amount 
                ELSE 0 END) AS no_group_amount
    FROM transactions as t
    LEFT JOIN promo as p
        ON t.customer_id = p.customer_id
    WHERE purchase_date BETWEEN '2024-02-01' AND '2024-02-14'
    GROUP BY t.purchase_date
    ORDER BY purchase_date;

--------------------------------------------------------------------------------------------------------------
-- 2. Engagement
-- 2.1 (KPI) Increase in Customer Engagement during Promotional Period
    -- Creating a set of variables for the KPI calculation
    WITH 
    sent_customers_that_made_purchase_during_promotion AS (
        SELECT COUNT(customer_id) AS total_customers
        FROM promo
        WHERE group="Sent"), 
    control_customers_that_made_purchase_during_promotion AS (
        SELECT COUNT(customer_id) AS total_customers
        FROM promo
        WHERE group="Control"), 
    all_sent_customers AS (
        SELECT COUNT(DISTINCT customer_id) AS total_customers
        FROM contact_history
        WHERE group="Sent"),
    all_control_customers AS (
        SELECT COUNT(DISTINCT customer_id) AS total_customers
        FROM contact_history
        WHERE group="Control"),
    percent_Sent_Customers_that_Made_Purchase_in_Promotional_Period AS (
        SELECT ROUND(100.0 * 
            (SELECT * FROM sent_customers_that_made_purchase_during_promotion)
            /
            (SELECT * FROM all_sent_customers), 2) as percentage),
    percent_Control_Customers_that_Made_Purchase_in_Promotional_Period AS (
        SELECT ROUND(100.0 * 
            (SELECT * FROM control_customers_that_made_purchase_during_promotion)
            /
            (SELECT * FROM all_control_customers), 2) as percentage)
    -- Final calculation for the KPI
    SELECT 
        ROUND(100.0 * 
            ((SELECT * FROM percent_Sent_Customers_that_Made_Purchase_in_Promotional_Period) - (SELECT * FROM percent_Control_Customers_that_Made_Purchase_in_Promotional_Period))
            /
            (SELECT * FROM percent_Control_Customers_that_Made_Purchase_in_Promotional_Period) 
            , 2) AS engagement_increase_percentage;

-- 2.2 Engagement  of Participants in Promotional Campaign by Group
    -- Variables calculated in 2.1
    WITH 
    sent_customers_that_made_purchase_during_promotion AS (
        SELECT COUNT(customer_id) AS total_customers
        FROM promo
        WHERE group="Sent"), 
    control_customers_that_made_purchase_during_promotion AS (
        SELECT COUNT(customer_id) AS total_customers
        FROM promo
        WHERE group="Control"), 
    all_sent_customers AS (
        SELECT COUNT(DISTINCT customer_id) AS total_customers
        FROM contact_history
        WHERE group="Sent"),
    all_control_customers AS (
        SELECT COUNT(DISTINCT customer_id) AS total_customers
        FROM contact_history
        WHERE group="Control"),
    percent_Sent_Customers_that_Made_Purchase_in_Promotional_Period AS (
        SELECT ROUND(100.0 * 
            (SELECT * FROM sent_customers_that_made_purchase_during_promotion)
            /
            (SELECT * FROM all_sent_customers), 2) as percentage),
    percent_Control_Customers_that_Made_Purchase_in_Promotional_Period AS (
        SELECT ROUND(100.0 * 
            (SELECT * FROM control_customers_that_made_purchase_during_promotion)
            /
            (SELECT * FROM all_control_customers), 2) as percentage)
    -- Final Select to show percentages for both groups      
    SELECT 'Control' AS group,
        (SELECT * FROM percent_Control_Customers_that_Made_Purchase_in_Promotional_Period) as percentage
    UNION ALL
    SELECT 'Sent' AS group,
        (SELECT * FROM percent_Sent_Customers_that_Made_Purchase_in_Promotional_Period) as percentage;

-- 2.3 Number of Purchases by Customers from Promotional Campaign
    SELECT t.purchase_date,
    count(CASE WHEN p.group = 'Sent' THEN t.customer_id
            ELSE null END) AS sent_customers,
    count(CASE WHEN p.group = 'Control' THEN t.customer_id
            ELSE null END) AS control_customers
    FROM transactions as t
    LEFT JOIN promo as p
        ON t.customer_id = p.customer_id
    WHERE purchase_date BETWEEN '2024-02-01' AND '2024-02-14'
    GROUP BY t.purchase_date
    ORDER BY purchase_date;

--------------------------------------------------------------------------------------------------------------
-- 3. Sales
-- 3.1 Average Amount Spent by Sent Participant
    SELECT AVG(sum_purchase_amount) AS avg_amount_spent_sent
    FROM promo
    WHERE group = 'Sent';

-- 3.2 Average Amount Spent by Control Participant
    SELECT AVG(sum_purchase_amount) AS avg_amount_spent_control
    FROM promo
    WHERE group = 'Control';

-- 3.3 (KPI) % Change in Average Amount Spent between Sent and Control Groups
    SELECT 
        ROUND(100.0 * 
            ((SELECT AVG(sum_purchase_amount) FROM promo WHERE group = 'Sent') - (SELECT AVG(sum_purchase_amount) FROM promo WHERE group = 'Control'))
            /
            (SELECT AVG(sum_purchase_amount) FROM promo WHERE group = 'Control') 
            , 2) AS avg_amount_spent_diff_percentage;

-- 3.4 Total Amount Spent by Sent Participants
    SELECT SUM(sum_purchase_amount) AS total_amount_spent_sent
    FROM promo
    WHERE group = 'Sent';

-- 3.5 (KPI) Estimated Increase in Total Amount Spent
    -- CTEs with measures calculated in 3.3 and 3.4
    WITH 
        Total_Amount_Spent_by_Sent_Participants AS(
            SELECT SUM(sum_purchase_amount) AS total_amount_spent_sent
            FROM promo
            WHERE group = 'Sent'), 
        Change_in_Average_Amount_Spent_between_Sent_and_Control_Groups AS(
            (SELECT 
                    ((SELECT AVG(sum_purchase_amount) FROM promo WHERE group = 'Sent') - (SELECT AVG(sum_purchase_amount) FROM promo WHERE group = 'Control'))
                    /
                    (SELECT AVG(sum_purchase_amount) FROM promo WHERE group = 'Control') 
                    AS avg_amount_spent_diff_percentage)
        )
    -- Final Select using CTEs
    SELECT (
        ROUND(
            -- Total Amount Spent by Sent Participants
            (SELECT * FROM Total_Amount_Spent_by_Sent_Participants)
            /
            -- 1 + Change in Average Amount Spent between Sent and Control Groups
            (1+ (SELECT * FROM Change_in_Average_Amount_Spent_between_Sent_and_Control_Groups))
            *
            -- Change in Average Amount Spent between Sent and Control Groups
            (SELECT * FROM Change_in_Average_Amount_Spent_between_Sent_and_Control_Groups)
        , 0)
    ) AS Calculated_Increase_in_Total_Amount_Spent;
-- 3.6. Sum of Purchase Amount by Customers from Promotional Campaign 
    SELECT group,
        COUNT(CASE WHEN sum_purchase_amount < 25 THEN 1 ELSE NULL END) AS `0-25`,
        COUNT(CASE WHEN sum_purchase_amount < 50 AND sum_purchase_amount >= 25 THEN 1 ELSE NULL END) AS `25-50`,
        COUNT(CASE WHEN sum_purchase_amount < 75 AND sum_purchase_amount >= 50 THEN 1 ELSE NULL END) AS `50-75`,
        COUNT(CASE WHEN sum_purchase_amount < 100 AND sum_purchase_amount >= 75 THEN 1 ELSE NULL END) AS `75-100`,  
        COUNT(CASE WHEN sum_purchase_amount < 125 AND sum_purchase_amount >= 100 THEN 1 ELSE NULL END) AS `100-125`,
        COUNT(CASE WHEN sum_purchase_amount < 150 AND sum_purchase_amount >= 125 THEN 1 ELSE NULL END) AS `125-150`,
        COUNT(CASE WHEN sum_purchase_amount < 175 AND sum_purchase_amount >= 150 THEN 1 ELSE NULL END) AS `150-175`,
        COUNT(CASE WHEN sum_purchase_amount < 200 AND sum_purchase_amount >= 175 THEN 1 ELSE NULL END) AS `175-200`,
        COUNT(CASE WHEN sum_purchase_amount < 225 AND sum_purchase_amount >= 200 THEN 1 ELSE NULL END) AS `200-225`,
        COUNT(CASE WHEN sum_purchase_amount < 250 AND sum_purchase_amount >= 225 THEN 1 ELSE NULL END) AS `225-250`,
        COUNT(CASE WHEN sum_purchase_amount < 300 AND sum_purchase_amount >= 250 THEN 1 ELSE NULL END) AS `250-300`,
        COUNT(CASE WHEN sum_purchase_amount >= 300 THEN 1 ELSE NULL END) AS `above_300`
    FROM promo
    GROUP BY group;
-- 3.7. Average Purchase Amount by Participant Group
    SELECT t.purchase_date,
        dayname(t.purchase_date) AS day_of_week,
        AVG(CASE WHEN p.group = 'Sent' THEN t.purchase_amount 
                ELSE null END) AS sent_amount,
        AVG(CASE WHEN p.group = 'Control' THEN t.purchase_amount 
                ELSE null END) AS control_amount
    FROM transactions as t
    LEFT JOIN promo as p
        ON t.customer_id = p.customer_id
    WHERE purchase_date BETWEEN '2024-02-01' AND '2024-02-14'
    GROUP BY t.purchase_date
    ORDER BY purchase_date;
